# Bone Fracture Detection - Comparative Analysis

This notebook provides a comprehensive comparative analysis of the bone fracture detection model, including:
- Training accuracy vs validation accuracy
- Metrics comparison (Precision, Recall, F1-Score)
- XAI (Explainable AI) comparison between different methods

In [3]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Import project modules
import sys
sys.path.append('..')
from src.model import FractureDetectionModel, get_transforms
from src.explain import Explainability
from src.utils import load_image, preprocess_image, predict
from train import BoneFractureDataset

ModuleNotFoundError: No module named 'torch'

## 2. Metrics Comparison

This section computes and compares model performance metrics: Precision, Recall, and F1-Score.

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model
model = FractureDetectionModel(num_classes=2)
model_path = '../models/fracture_detection_model.pth'

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("✅ Model loaded successfully!")
else:
    print("⚠️ Model not found. Please train the model first.")

model = model.to(device)
model.eval()

In [ ]:
# Evaluate model on test set and compute metrics
transform = get_transforms()

# Load test dataset
test_dataset = BoneFractureDataset('../model/testing', transform)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"Test dataset size: {len(test_dataset)}")

# Collect predictions and ground truth
all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

In [ ]:
# Compute metrics
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)
accuracy = np.mean(all_predictions == all_labels)

print(f"Model Performance Metrics:")
print(f"- Accuracy: {accuracy:.4f}")
print(f"- Precision: {precision:.4f}")
print(f"- Recall: {recall:.4f}")
print(f"- F1-Score: {f1:.4f}")

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_predictions)
print(f"\nConfusion Matrix:")
print(cm)

In [ ]:
# Plot Metrics Comparison Bar Chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy * 100, precision * 100, recall * 100, f1 * 100]

plt.figure(figsize=(10, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
bars = plt.bar(metrics, values, color=colors, edgecolor='black', linewidth=1.5)

plt.xlabel('Metrics', fontsize=12)
plt.ylabel('Score (%)', fontsize=12)
plt.title('Model Performance Metrics Comparison', fontsize=14, fontweight='bold')
plt.ylim(0, 100)

# Add value labels on bars
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{val:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()

# Save metrics comparison graph
os.makedirs('../model', exist_ok=True)
plt.savefig('../model/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Metrics comparison graph saved to model/metrics_comparison.png")

## 3. XAI (Explainable AI) Comparison

This section compares different Explainable AI methods:
- **Grad-CAM**: Gradient-weighted Class Activation Mapping
- **LIME**: Local Interpretable Model-agnostic Explanations

In [ ]:
# Select a sample image for XAI demonstration
test_images_dir = '../model/testing'
fractured_dir = os.path.join(test_images_dir, 'fractured')
not_fractured_dir = os.path.join(test_images_dir, 'not_fractured')

# Get sample fractured image
sample_images = []
if os.path.exists(fractured_dir):
    fractured_images = [os.path.join(fractured_dir, f) for f in os.listdir(fractured_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    sample_images.extend(fractured_images[:2])  # Take 2 fractured images

if os.path.exists(not_fractured_dir):
    not_fractured_images = [os.path.join(not_fractured_dir, f) for f in os.listdir(not_fractured_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    sample_images.extend(not_fractured_images[:2])  # Take 2 non-fractured images

print(f"Selected {len(sample_images)} sample images for XAI comparison")

In [ ]:
# Initialize explainer
explainer = Explainability(model, device)

# Function for LIME prediction
def lime_predict(images):
    """Prediction function for LIME"""
    predictions = []
    for img in images:
        img_pil = Image.fromarray(img.astype('uint8'))
        img_resized = img_pil.resize((224, 224))
        img_tensor = transform(img_resized).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output = model(img_tensor)
            probs = torch.nn.functional.softmax(output, dim=1)
        predictions.append(probs.cpu().numpy())
    return np.array(predictions).squeeze()

In [ ]:
# Generate XAI comparisons for each sample image
fig, axes = plt.subplots(len(sample_images), 3, figsize=(15, 5 * len(sample_images)))

if len(sample_images) == 1:
    axes = axes.reshape(1, -1)

for idx, img_path in enumerate(sample_images):
    print(f"Processing image {idx + 1}/{len(sample_images)}: {os.path.basename(img_path)}")
    
    # Load and preprocess image
    original_image = Image.open(img_path).convert('RGB')
    original_image_resized = original_image.resize((224, 224))
    input_tensor = transform(original_image_resized).unsqueeze(0).to(device)
    
    # Get prediction
    predicted_class, confidence = predict(model, input_tensor, device)
    class_names = ['Not Fractured', 'Fractured']
    
    # Grad-CAM
    grad_cam_attr = explainer.grad_cam(input_tensor, predicted_class)
    heatmap_gradcam = explainer.generate_heatmap(grad_cam_attr, np.array(original_image_resized))
    
    # LIME
    try:
        lime_exp = explainer.lime_explanation(
            np.array(original_image_resized), 
            lime_predict, 
            num_samples=500
        )
        lime_temp, lime_mask = lime_exp.get_image_and_mask(
            lime_exp.top_labels[0], 
            positive_only=True, 
            num_features=10
        )
        lime_visualization = mark_boundaries(lime_temp / 255.0, lime_mask)
    except Exception as e:
        print(f"LIME failed for this image: {e}")
        lime_visualization = np.array(original_image_resized) / 255.0
    
    # Display original image
    axes[idx, 0].imshow(original_image_resized)
    axes[idx, 0].set_title(f'Original Image\nPred: {class_names[predicted_class]} ({confidence:.2f})')
    axes[idx, 0].axis('off')
    
    # Display Grad-CAM
    axes[idx, 1].imshow(heatmap_gradcam)
    axes[idx, 1].set_title('Grad-CAM')
    axes[idx, 1].axis('off')
    
    # Display LIME
    axes[idx, 2].imshow(lime_visualization)
    axes[idx, 2].set_title('LIME Explanation')
    axes[idx, 2].axis('off')

plt.suptitle('XAI Methods Comparison: Grad-CAM vs LIME', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

# Save XAI comparison graph
plt.savefig('../model/xai_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ XAI comparison graph saved to model/xai_comparison.png")

## Summary

This notebook provides a comprehensive analysis of the bone fracture detection model:

1. **Accuracy Graph**: Shows training vs validation accuracy over epochs
2. **Metrics Comparison**: Displays Precision, Recall, F1-Score, and Accuracy metrics
3. **XAI Comparison**: Compares Grad-CAM and LIME explainability methods

All graphs are saved in the `model/` directory.